# Veyra · Colab Backup Builder 3
### Stable or Preview · Your existing release key · Verified APK outputs

**Made by Bhavishya Dalal.** This is the interactive Colab alternative to GitHub Actions. It does not change or interrupt an Actions run.

1. Upload this notebook to [Google Colab](https://colab.research.google.com/), save a **private** copy, and use a CPU runtime.
2. Run the helper cells at the top. In **Step 1**, choose Stable **0.1.8** or Preview **0.1.9-beta1**, your architecture, and signing mode. Review the SDK licence terms and tick the checkbox if you agree.
3. Run Steps 2–4. Upload **Veyra_PRIVATE_Signing_Backup.zip** in Step 4. No manual extraction or GitHub secrets are needed in Colab.
4. Run Step 5. Download the APK and results in Step 7 before the runtime expires. Drive backup is optional.

| Default | Value |
| --- | --- |
| Source | Stable 0.1.8 (completed preview selectable) |
| APK | Production Release · ARM64 |
| Signing | Existing key; checked against the published 0.1.5 certificate |
| Optimization | R8 + resource shrinking; no debug fallback |
| Resources | Java 17 · 2 workers · 3 GiB heap |
| Tests / Drive | Off; optional |

**New:** preset source checksums, source-ZIP upload or custom GitHub URL, nested signing-backup support, separate key-password prompt when missing, expected package/native-ABI checks, and bounded duplicate-safe ZIP handling. Presets retain the source version; this builder never relabels beta as stable. Android SDK installation uses `platform-tools`, never the retired `tools` package.

Colab uploads your private signing material to your own remote runtime; it is **not offline signing**. Do not share private backups or executed notebooks with sensitive outputs. Credentials and keys are excluded from the public results ZIP. This notebook has been checked locally with synthetic inputs, not used here to build an APK or run a hosted Colab session.


## Builder helpers
Run this cell as part of **Run all**. It defines the download, SDK, signing, build and verification functions. No APK build starts here.

In [ ]:
from pathlib import Path, PurePosixPath
from collections import deque
import base64, datetime, getpass, hashlib, html, json, os, platform, queue, re
import secrets, shutil, signal, stat, subprocess, threading, time, urllib.parse, urllib.request
import uuid, zipfile, xml.etree.ElementTree as ET

DEFAULT_URL = 'https://github.com/Bhavishyadalal/Veyra/raw/refs/heads/main/downloads/Veyra_Android_Source_0.1.8_Stable.zip'
PREVIEW_URL = 'https://github.com/Bhavishyadalal/Veyra/raw/refs/heads/preview/0.1.9-beta1/downloads/Veyra_Android_Source_0.1.9-beta1.zip'
SECRETS = []
LAST_TASKS = {}

def clean(text):
    text = str(text)
    for value in sorted(set(SECRETS), key=len, reverse=True):
        if value:
            text = text.replace(value, '[REDACTED]')
    return text

def sha(path, algorithm='sha256'):
    h = hashlib.new(algorithm)
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def run(command, label, env=None, cwd=None, timeout=7200, stdin=None, check=True, quiet=False):
    """Stream output, retain a log and show a heartbeat during quiet Gradle work."""
    global LAST_TASKS
    LAST_TASKS = {}
    current_task = 'Starting process'
    last_output = time.monotonic()
    print('\n▶ ' + label, flush=True)
    command = [str(x) for x in command]
    log = RUN / 'build.log'
    lines, tail, messages = [], deque(maxlen=25), queue.Queue(maxsize=2000)
    start = last_heartbeat = time.monotonic()
    process = subprocess.Popen(command, cwd=cwd, env=env, stdin=subprocess.PIPE if stdin is not None else subprocess.DEVNULL,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace',
                               bufsize=1, start_new_session=True)
    def reader():
        for line in process.stdout:
            messages.put(line)
        messages.put(None)
    threading.Thread(target=reader, daemon=True).start()
    if stdin is not None:
        def feed():
            try:
                process.stdin.write(stdin)
                process.stdin.close()
            except (BrokenPipeError, OSError):
                pass
        threading.Thread(target=feed, daemon=True).start()
    done = False
    try:
        with log.open('a', encoding='utf-8') as output:
            output.write('\n=== ' + label + ' ===\n')
            while not done:
                try:
                    line = messages.get(timeout=1)
                    if line is None:
                        done = True
                    else:
                        line = clean(line)
                        last_output = time.monotonic()
                        task = re.search(r'> Task (:[^\s]+)(.*)', line)
                        if task:
                            current_task = task[1]
                            status = task[2].strip().split()
                            LAST_TASKS[current_task] = status[0] if status else 'EXECUTED'
                        output.write(line); output.flush()
                        lines.append(line); tail.append(line.rstrip())
                        if not quiet:
                            print(line, end='', flush=True)
                except queue.Empty:
                    pass
                now = time.monotonic()
                if process.poll() is None and now - last_heartbeat >= 30:
                    elapsed = int(now-start)
                    message = f'… {elapsed//60}m {elapsed%60}s elapsed | PID {process.pid} running | last task: {current_task} | no output for {int(now-last_output)}s. This is activity monitoring, not a completion estimate.'
                    print(message, flush=True); output.write(message+'\n'); output.flush()
                    last_heartbeat = now
                if now - start > timeout:
                    raise TimeoutError(f'{label} exceeded its time limit. See build.log.')
            code = process.wait(timeout=20)
    except BaseException:
        if process.poll() is None:
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL); process.wait()
        raise
    text = ''.join(lines)
    if check and code:
        if quiet:
            print('\n'.join(tail))
        raise RuntimeError(f'{label} failed (exit {code}). Download build.log from the results cell.')
    return code, text

def download(url, destination, digest=None, algorithm='sha256', fresh=False):
    destination = Path(destination); destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file() and digest and not fresh and sha(destination, algorithm) == digest.lower():
        print('Reusing verified download: '+destination.name); return destination
    partial = destination.with_name(destination.name+'.part')
    for attempt in range(3):
        try:
            request = urllib.request.Request(url, headers={'User-Agent':'Veyra-Colab-Builder/3.0'})
            with urllib.request.urlopen(request, timeout=90) as response, partial.open('wb') as output:
                total = int(response.headers.get('Content-Length') or 0); done = 0; previous = 0
                if total > 3*1024**3: raise RuntimeError('Download exceeds 3 GiB limit.')
                while True:
                    block = response.read(1024*1024)
                    if not block: break
                    output.write(block); done += len(block)
                    if done > 3*1024**3: raise RuntimeError('Download exceeds 3 GiB limit.')
                    if time.monotonic()-previous > 5:
                        suffix = f' / {total/1024**2:.1f} MiB' if total else ' MiB'
                        print(f'↓ {destination.name}: {done/1024**2:.1f}'+suffix, flush=True)
                        previous = time.monotonic()
            if total and done != total: raise RuntimeError('Incomplete HTTP download.')
            if not partial.stat().st_size:
                raise RuntimeError('The server returned an empty file.')
            if digest and sha(partial, algorithm) != digest.lower():
                raise RuntimeError('Download checksum mismatch: '+destination.name)
            partial.replace(destination); return destination
        except Exception as exc:
            partial.unlink(missing_ok=True)
            if attempt == 2:
                raise RuntimeError('Download failed: '+destination.name+' — '+clean(exc)) from None
            print(f'Download retry {attempt+2}/3…', flush=True); time.sleep(2**attempt)

def extract_zip(archive, destination, limit=3*1024**3):
    destination = Path(destination).resolve(); destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        if len(z.infolist()) > 100000 or sum(i.file_size for i in z.infolist()) > limit:
            raise ValueError('Archive exceeds extraction limits.')
        targets=set()
        for item in z.infolist():
            name = item.filename.replace('\\','/'); parts = PurePosixPath(name).parts
            if name.startswith('/') or '..' in parts or (parts and ':' in parts[0]):
                raise ValueError('Unsafe archive path: '+name)
            if stat.S_ISLNK(item.external_attr >> 16):
                raise ValueError('Symlink entry is not supported: '+name)
            target = (destination/name).resolve()
            if target in targets: raise ValueError('Duplicate archive destination.')
            targets.add(target)
            if not target.is_relative_to(destination):
                raise ValueError('Archive entry escapes its destination.')
        if z.testzip() is not None:
            raise ValueError('ZIP integrity check failed.')
        for item in z.infolist():
            name = item.filename.replace('\\','/'); target = destination/name
            if item.is_dir(): target.mkdir(parents=True, exist_ok=True); continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(item) as source, target.open('wb') as output:
                shutil.copyfileobj(source, output)
            mode = (item.external_attr >> 16) & 0o777
            if mode: target.chmod(mode)

def normalize_url(value):
    p = urllib.parse.urlsplit(value.strip())
    if p.scheme != 'https' or p.hostname not in {'github.com','raw.githubusercontent.com','codeload.github.com'} or p.username:
        raise ValueError('Paste a public HTTPS GitHub source ZIP URL. Private repositories need a separate authenticated workflow.')
    if p.hostname == 'github.com' and '/blob/' in p.path:
        p = p._replace(path=p.path.replace('/blob/','/raw/',1))
    return urllib.parse.urlunsplit(p)

def project_root(destination):
    choices = [p.parent for p in Path(destination).rglob('gradlew')
               if (p.parent/'app/build.gradle.kts').is_file() and
               any((p.parent/n).is_file() for n in ['settings.gradle.kts','settings.gradle'])]
    if len(choices) != 1:
        raise ValueError('Expected one Veyra Android project containing app/, gradlew and settings.gradle.kts.')
    return choices[0]

def source_fingerprint(project):
    result={}
    for p in project.rglob('*'):
        if not p.is_file(): continue
        rel=p.relative_to(project)
        if any(x in {'.gradle','build','node_modules'} for x in rel.parts): continue
        if 'src' in rel.parts or p.name.endswith('.gradle.kts') or p.name in {'gradle.properties','gradle-wrapper.properties'}:
            result[rel.as_posix()]=sha(p)
    return result

def detect(project):
    root = (project/'build.gradle.kts').read_text(); app = (project/'app/build.gradle.kts').read_text()
    def extra(name, default=''):
        m = re.search(r'extra\["'+name+r'"\]\s*=\s*(?:"([^"]*)"|(\d+))',root)
        return next((x for x in m.groups() if x is not None), default) if m else default
    platforms, ndks, cmakes = set(), set(), set()
    for p in project.rglob('build.gradle.kts'):
        if any(x in {'.gradle','build','node_modules'} for x in p.relative_to(project).parts): continue
        s = re.sub(r'(?m)^\s*//.*$', '', p.read_text())
        platforms.update('platforms;android-'+n for n in re.findall(r'compileSdk\s*=\s*(\d+)',s))
        platforms.update('platforms;android-'+n for n in re.findall(r'compileSdkVersion\(\s*(\d+)',s))
        for line in s.splitlines():
            if 'compileSdkVersion(' in line:
                platforms.update('platforms;android-'+n for n in re.findall(r'android-([\d.]+)',line))
        ndks.update(re.findall(r'ndkVersion\s*=\s*"([\d.]+)"',s))
        cmakes.update(re.findall(r'cmake\s*\{[^}]*?version\s*=\s*"([\d.]+)"',s,re.S))
    if not platforms: raise ValueError('Could not determine compile SDK. Inspect the source build files.')
    explicit_bt = re.search(r'buildToolsVersion\s*(?:=|\()?\s*"([\d.]+)"',app)
    bt = BUILD_TOOLS_OVERRIDE.strip() or (explicit_bt[1] if explicit_bt else '35.0.0')
    packages = ['platform-tools','build-tools;'+bt]+sorted(platforms)+['ndk;'+n for n in sorted(ndks)]+['cmake;'+n for n in sorted(cmakes)]
    if any(not re.fullmatch(r'(platform-tools|(?:build-tools|ndk|cmake);[\d.]+|platforms;android-[\d.]+)',x) for x in packages):
        raise ValueError('Invalid detected Android package version.')
    version=extra('veyraVersionName','unknown'); suffix=extra('veyraBetaSuffix')
    return {'version':version+('-'+suffix if suffix else ''), 'version_code':extra('veyraVersionCode'),
            'packages':packages,'build_tools':bt}

def flatten_package(path):
    """Repair only an extra archive-root directory; never invent SDK metadata."""
    path=Path(path)
    if (path/'source.properties').is_file(): return
    candidates=[p.parent for p in path.glob('*/source.properties')]
    if len(candidates)!=1: return
    nested=candidates[0]
    conflicts=[p.name for p in nested.iterdir() if (path/p.name).exists()]
    if conflicts: raise RuntimeError('Nested SDK package has conflicting files: '+', '.join(conflicts))
    for p in list(nested.iterdir()): shutil.move(str(p),str(path/p.name))
    nested.rmdir(); print('Repaired extra directory level: '+path.name)

def choose_java():
    for p in sorted(Path('/usr/lib/jvm').glob('*')):
        if (p/'bin/javac').is_file() and re.search(rf'\b{JAVA_VERSION}\b',p.name): return p
    prefix=[] if os.geteuid()==0 else ['sudo']
    run(prefix+['apt-get','update','-qq'],'Refresh package index',timeout=900,quiet=True)
    run(prefix+['apt-get','install','-y','-qq',f'openjdk-{JAVA_VERSION}-jdk-headless'],'Install Java compiler',timeout=1800,quiet=True)
    for p in sorted(Path('/usr/lib/jvm').glob('*')):
        if (p/'bin/javac').is_file() and re.search(rf'\b{JAVA_VERSION}\b',p.name): return p
    raise RuntimeError('JDK with javac was not found after installation.')


In [ ]:
def setup_tools():
    global JAVA, SDK, ENV, SDK_XML
    if not ACCEPT_ANDROID_LICENSES:
        raise ValueError('Tick the Android SDK licence checkbox in the configuration cell, rerun it, then continue.')
    if platform.system()!='Linux' or platform.machine() not in {'x86_64','AMD64'}:
        raise RuntimeError('Use a hosted Colab Linux CPU runtime.')
    if shutil.disk_usage(BASE).free < 12*1024**3:
        raise RuntimeError('At least 12 GiB free disk space is required for a fresh build.')
    JAVA=choose_java(); SDK=TOOLS/'android-sdk'; SDK.mkdir(parents=True,exist_ok=True)
    ENV=dict(os.environ, JAVA_HOME=str(JAVA),ANDROID_HOME=str(SDK),ANDROID_SDK_ROOT=str(SDK),
             ANDROID_USER_HOME=str(TOOLS/'android-user'),GRADLE_USER_HOME=str(TOOLS/'gradle-cache'))
    ENV['PATH']=str(JAVA/'bin')+os.pathsep+ENV['PATH']
    # Hosted Colab uses normal internet access; do not import another machine's JVM/proxy flags.
    for name in ['JAVA_TOOL_OPTIONS','JAVA_OPTS','GRADLE_OPTS']: ENV[name]=''
    metadata=download('https://dl.google.com/android/repository/repository2-1.xml',RUN/'android-repository.xml',fresh=True)
    SDK_XML=ET.parse(metadata).getroot()
    cli=SDK/'cmdline-tools/19.0'
    if not (cli/'lib/sdkmanager-classpath.jar').is_file():
        package=next((p for p in SDK_XML.findall('remotePackage') if p.get('path')=='cmdline-tools;19.0'),None)
        if package is None: raise RuntimeError('Pinned command-line tools 19.0 are not listed by Google.')
        archive=next(a for a in package.findall('archives/archive') if a.findtext('host-os')=='linux')
        name=archive.findtext('complete/url'); checksum=archive.find('complete/checksum')
        digest=checksum.text.strip(); algo=checksum.get('type') or ('sha256' if len(digest)==64 else 'sha1')
        package_file=download('https://dl.google.com/android/repository/'+name,TOOLS/'downloads'/Path(name).name,digest,algo)
        unpack=TOOLS/('cli-unpack-'+uuid.uuid4().hex[:8]); extract_zip(package_file,unpack)
        source=unpack/'cmdline-tools'
        if cli.exists(): shutil.rmtree(cli)
        cli.parent.mkdir(parents=True,exist_ok=True); shutil.move(str(source),str(cli)); shutil.rmtree(unpack)
    sdk_command=[JAVA/'bin/java','-Dcom.android.sdklib.toolsdir='+str(cli),'-cp',cli/'lib/sdkmanager-classpath.jar',
                 'com.android.sdklib.tool.sdkmanager.SdkManagerCli','--sdk_root='+str(SDK)]
    run(sdk_command+['--licenses'],'Accept Android SDK licences',env=ENV,stdin='y\n'*200,timeout=900,quiet=True)
    run(sdk_command+INFO['packages'],'Install/reuse project Android packages',env=ENV,stdin='y\n'*200,timeout=3600)
    for name in INFO['packages']:
        if name=='platform-tools':
            if not (SDK/'platform-tools/adb').is_file(): raise RuntimeError('Missing platform-tools.')
            continue
        path=SDK.joinpath(*name.split(';')); flatten_package(path)
        if not (path/'source.properties').is_file(): raise RuntimeError('Incomplete Android package: '+name)
        if name.startswith('platforms;') and not (path/'android.jar').is_file(): raise RuntimeError('Missing android.jar: '+name)
        if name.startswith('ndk;') and not (path/'toolchains/llvm/prebuilt/linux-x86_64/bin/clang').exists(): raise RuntimeError('Incomplete NDK compiler.')
        if name.startswith('cmake;') and not (path/'bin/cmake').is_file(): raise RuntimeError('Incomplete CMake installation.')
    print('Tools ready: JDK '+str(JAVA_VERSION)+'; project SDK packages verified.')


In [ ]:
def backup_contents(path):
    with zipfile.ZipFile(path) as z:
        items=z.infolist()
        if len(items)>64 or sum(i.file_size for i in items)>16*1024**2:
            raise ValueError('Signing backup exceeds size/entry limits.')
        names=set()
        for item in items:
            name=item.filename
            if name in names or name.startswith('/') or '\\' in name or '..' in PurePosixPath(name).parts or ':' in name or stat.S_ISLNK(item.external_attr>>16):
                raise ValueError('Unsafe or duplicate signing archive path.')
            names.add(name)
        choices=[i for i in items if not i.is_dir() and i.filename.lower().endswith(('.jks','.keystore'))]
        jsons=[i for i in items if re.search(r'(^|/)signing/credentials\.json$',i.filename)]
        if len(choices)!=1 or not 0<choices[0].file_size<=8*1024**2 or len(jsons)>1:
            raise ValueError('Signing ZIP must contain exactly one nonempty JKS/keystore and at most one credentials.json.')
        credentials={}
        if jsons:
            if jsons[0].file_size>65536: raise ValueError('Invalid signing credentials file.')
            credentials=json.loads(z.read(jsons[0]))
            if not isinstance(credentials,dict) or any(k in credentials and (not isinstance(credentials[k],str) or not credentials[k]) for k in ['alias','store_password','key_password']):
                raise ValueError('Signing credential fields must be nonempty strings.')
        return z.read(choices[0]),credentials

def download_button(path, title):
    from google.colab import files
    import ipywidgets as widgets
    from IPython.display import display
    button=widgets.Button(description=title,layout=widgets.Layout(width='auto'),button_style='primary')
    output=widgets.Output()
    def clicked(_):
        with output:
            files.download(str(path))
    button.on_click(clicked); display(button,output)

def configure_signing():
    global SIGNING, KEY_ENV, NEW_KEY_BACKUP
    SIGNING=None; KEY_ENV={}; NEW_KEY_BACKUP=None
    if SIGNING_MODE not in {'Upload existing key or backup','Create new key (new installs)','Unsigned release'}:
        raise ValueError('Choose one of the three supported signing modes.')
    if SIGNING_MODE=='Unsigned release':
        print('Unsigned mode: the APK will NOT be installable until signed.'); return
    from google.colab import files
    private=RUN/'private'; private.mkdir(mode=0o700,exist_ok=True)
    key=private/'signing.jks'; credentials={}
    if SIGNING_MODE=='Upload existing key or backup':
        print('Upload one .jks/.keystore or your portable Veyra signing-backup ZIP.')
        # Upload to the private working directory; never to the project or public output folder.
        previous=Path.cwd()
        try:
            os.chdir(private); uploaded=files.upload()
        finally:
            os.chdir(previous)
        for saved in private.iterdir():
            if saved.is_file(): saved.chmod(0o600)
        if len(uploaded)!=1: raise ValueError('Upload exactly one signing file.')
        filename,raw=next(iter(uploaded.items()))
        upload=private/Path(filename).name; upload.write_bytes(raw); upload.chmod(0o600)
        if zipfile.is_zipfile(upload):
            raw,credentials=backup_contents(upload)
        elif len(raw)>8*1024**2:
            raise ValueError('Signing key is too large.')
        key.write_bytes(raw); key.chmod(0o600)
        alias=credentials.get('alias') or input('Key alias: ').strip()
        store=credentials.get('store_password') or getpass.getpass('Keystore password (hidden): ')
        password=credentials.get('key_password') or getpass.getpass('Key password (Enter = keystore password): ') or store
        del raw,uploaded,credentials
    else:
        alias='veyra'; store=getpass.getpass('New key password (Enter = generate securely): ') or secrets.token_urlsafe(24); password=store
    if not alias or not store or not password: raise ValueError('Alias and signing passwords are required.')
    SECRETS.extend([store,password]); KEY_ENV={'VEYRA_STORE_PASSWORD':str(store),'VEYRA_KEY_PASSWORD':str(password),
                                            'VEYRA_KEY_ALIAS':str(alias),'VEYRA_KEY_PATH':str(key)}
    signing_env=dict(ENV,**KEY_ENV)
    if SIGNING_MODE=='Create new key (new installs)':
        run([JAVA/'bin/keytool','-genkeypair','-keystore',key,'-storetype','JKS','-alias',alias,'-keyalg','RSA','-keysize','3072',
             '-validity','10000','-dname','CN=Veyra Release','-storepass:env','VEYRA_STORE_PASSWORD',
             '-keypass:env','VEYRA_KEY_PASSWORD','-noprompt'],'Create a NEW signing identity',env=signing_env,quiet=True)
        backup=private/'Veyra_SIGNING_BACKUP_KEEP_PRIVATE.zip'
        with zipfile.ZipFile(backup,'w',zipfile.ZIP_DEFLATED) as z:
            z.write(key,'signing/builder.jks')
            z.writestr('signing/credentials.json',json.dumps({'alias':alias,'store_password':store,'key_password':password}))
        backup.chmod(0o600); NEW_KEY_BACKUP=backup
        print('Save this PRIVATE backup for every future update. It contains the key and passwords; do not publish it.')
        download_button(backup,'Download PRIVATE signing backup')
    run([JAVA/'bin/keytool','-list','-keystore',key,'-alias',alias,'-storepass:env','VEYRA_STORE_PASSWORD'],
        'Validate signing key and alias',env=signing_env,quiet=True)
    proof=private/'key-validation.csr'
    try:
        run([JAVA/'bin/keytool','-certreq','-keystore',key,'-alias',alias,
             '-storepass:env','VEYRA_STORE_PASSWORD','-keypass:env','VEYRA_KEY_PASSWORD','-file',proof],
            'Validate private-key password',env=signing_env,quiet=True)
    finally: proof.unlink(missing_ok=True)
    cert_file=private/'certificate.der'
    run([JAVA/'bin/keytool','-exportcert','-keystore',key,'-alias',alias,
         '-storepass:env','VEYRA_STORE_PASSWORD','-file',cert_file],
        'Read selected signing certificate',env=signing_env,quiet=True)
    fingerprint=sha(cert_file)
    expected=EXPECTED_CERT_SHA256.replace(':','').strip().lower()
    if expected and fingerprint!=expected:
        raise RuntimeError('The selected key does not match the expected installed-app certificate.')
    SIGNING={'alias':alias,'path':key,'certificate_sha256':fingerprint}
    print('Selected key certificate SHA-256:',fingerprint)
    print('Signing key loaded. Passwords were not printed or embedded in this notebook.')


In [ ]:
INIT_SCRIPT = '''allprojects { p ->
 p.plugins.withId('com.android.application') {
  p.extensions.getByName('androidComponents').finalizeDsl { a ->
   def releaseType = a.buildTypes.getByName('release')
   releaseType.minifyEnabled = true
   releaseType.shrinkResources = true
   def rules = releaseType.proguardFiles.findAll { it.name != 'proguard-android.txt' }
   def optimized = a.getDefaultProguardFile('proguard-android-optimize.txt')
   if (!rules.any { it.canonicalFile == optimized.canonicalFile }) rules.add(optimized)
   releaseType.setProguardFiles(rules)
   def abi = System.getenv('VEYRA_TARGET_ABI')
   if (a.splits.abi.enable && abi) {
    if (abi == 'universal') { a.splits.abi.universalApk = true }
    else { a.splits.abi.reset(); a.splits.abi.include(abi); a.splits.abi.universalApk = false }
   }
   if (System.getenv('VEYRA_KEY_PATH')) {
    def s = a.signingConfigs.maybeCreate('veyraColab')
    s.storeFile = new File(System.getenv('VEYRA_KEY_PATH'))
    s.storePassword = System.getenv('VEYRA_STORE_PASSWORD')
    s.keyAlias = System.getenv('VEYRA_KEY_ALIAS')
    s.keyPassword = System.getenv('VEYRA_KEY_PASSWORD')
    a.buildTypes.getByName('release').signingConfig = s
   } else {
    a.buildTypes.getByName('release').signingConfig = null
   }
  }
 }
}
'''

def zip_results():
    target=RUN/'Veyra_Build_Results.zip'
    with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file(): z.write(p,p.relative_to(OUTPUT).as_posix())
        if (RUN/'build.log').exists(): z.write(RUN/'build.log','build.log')
    return target

def build_apk():
    global RESULT, RESULT_ZIP
    RESULT={'status':'building','source_url':SOURCE_URL,'source_sha256':SOURCE_SHA,'version_detected':INFO['version'],
            'version_code_detected':INFO['version_code'],'variant':VARIANT,'architecture':ARCHITECTURE,
            'tests':{'status':'not_requested' if not RUN_TESTS else 'pending'},'artifacts':[]}
    (OUTPUT/'build-report.json').write_text(json.dumps(RESULT,indent=2))
    for old in OUTPUT.glob('*.apk'): old.unlink()
    (OUTPUT/'SHA256SUMS.txt').unlink(missing_ok=True)
    try:
        if not (PROJECT/'gradle/wrapper/gradle-wrapper.jar').is_file():
            raise RuntimeError('Missing Gradle wrapper JAR in source ZIP.')
        if SIGNING_MODE!='Unsigned release' and SIGNING is None:
            raise RuntimeError('Run the signing cell before building; no unsigned fallback is selected.')
        wrapper=PROJECT/'gradle/wrapper/gradle-wrapper.properties'; s=wrapper.read_text()
        match=re.search(r'^distributionUrl=(.+)$',s,re.M)
        if not match: raise RuntimeError('Gradle wrapper URL is missing.')
        url=match[1].strip().replace('\\:',':')
        if urllib.parse.urlsplit(url).hostname not in {'services.gradle.org','downloads.gradle.org'}:
            raise RuntimeError('This notebook expects an official Gradle wrapper distribution.')
        pinned=re.search(r'^distributionSha256Sum=([a-fA-F0-9]{64})\s*$',s,re.M)
        if not pinned: raise RuntimeError('Source wrapper has no valid SHA-256 pin. Add an official checksum in the source project before building.')
        local=PROJECT/'local.properties'; existing=local.read_text() if local.exists() else ''
        local.write_text(re.sub(r'^sdk\.dir=.*(?:\n|$)','',existing,flags=re.M)+'\nsdk.dir='+SDK.as_posix()+'\n')
        init=RUN/'signing.init.gradle'; init.write_text(INIT_SCRIPT)
        env=dict(ENV,**KEY_ENV)
        env['VEYRA_TARGET_ABI']={'ARM64 (my Realme phone)':'arm64-v8a','Universal (larger APK)':'universal','ARMv7':'armeabi-v7a','x86_64 (emulator)':'x86_64'}[ARCHITECTURE]
        wrapper_cmd=[JAVA/'bin/java','-cp',PROJECT/'gradle/wrapper/gradle-wrapper.jar','org.gradle.wrapper.GradleWrapperMain']
        common=['--no-daemon','--max-workers='+str(WORKERS),'--console=plain','--stacktrace',
                '-Dorg.gradle.java.home='+str(JAVA),'-Dorg.gradle.jvmargs=-Xmx'+str(HEAP_MB)+'m -Dfile.encoding=UTF-8',
                '-Pkotlin.compiler.execution.strategy=in-process','-I',init]
        r8_task=':app:minify'+VARIANT+'WithR8'
        run(wrapper_cmd+[':app:assemble'+VARIANT,r8_task]+common,'Build optimized '+VARIANT+' APK',env=env,cwd=PROJECT,timeout=BUILD_TIMEOUT_MINUTES*60)
        status=LAST_TASKS.get(r8_task)
        if status not in {'EXECUTED','UP-TO-DATE','FROM-CACHE'}:
            raise RuntimeError('Selected R8 task did not complete; no unoptimized fallback accepted.')
        reports=PROJECT/'app/build/outputs/mapping'/(VARIANT[:1].lower()+VARIANT[1:])
        for filename in ['mapping.txt','configuration.txt']:
            report=reports/filename
            if not report.is_file() or not report.stat().st_size:
                raise RuntimeError('Missing R8 report: '+filename)
        if re.search(r'^\s*-(?:dontoptimize|dontshrink)\b',(reports/'configuration.txt').read_text(),re.M):
            raise RuntimeError('Project rules globally disable R8 optimization/shrinking. Review them; keep rules were preserved.')
        saved=OUTPUT/'R8-reports'; saved.mkdir(exist_ok=True)
        for filename in ['mapping.txt','configuration.txt','usage.txt','seeds.txt','resources.txt']:
            if (reports/filename).is_file(): shutil.copy2(reports/filename,saved/filename)
        RESULT['optimization']={'mode':'release-r8','verified':True,'task':r8_task,'task_status':status,
                                'resource_shrinking_enabled':True,'mapping_sha256':sha(saved/'mapping.txt')}
        print('R8 verified. Effective configuration and mapping saved in result ZIP.')
        RESULT['assembly']='passed'
        if RUN_TESTS:
            test_results=PROJECT/'app/build/test-results'/('test'+VARIANT+'UnitTest')
            if test_results.exists(): shutil.rmtree(test_results)
            code,_=run(wrapper_cmd+[':app:test'+VARIANT+'UnitTest']+common,'Run existing unit tests',env=env,cwd=PROJECT,check=False,timeout=7200)
            counts={k:0 for k in ['tests','failures','errors','skipped']}
            for p in (PROJECT/'app/build/test-results'/('test'+VARIANT+'UnitTest')).glob('TEST-*.xml'):
                root=ET.parse(p).getroot()
                for k in counts: counts[k]+=int(root.get(k,0))
            RESULT['tests']={'status':'passed' if code==0 and counts['tests']>0 and not counts['failures'] and not counts['errors'] else 'failed_or_blocked',
                             'gradle_exit_code':code,**counts}
            if RESULT['tests']['status']!='passed':
                print('UNIT TESTS DID NOT PASS. APK assembly succeeded; see build.log. No tests were deleted or disabled.')
        tools=sorted([p for p in (SDK/'build-tools').iterdir() if (p/'lib/apksigner.jar').is_file() and (p/'aapt2').is_file()],
                     key=lambda p:tuple(int(x) for x in re.findall(r'\d+',p.name)))
        if not tools: raise RuntimeError('APK verification tools are missing.')
        bt=tools[-1]; flavor='prod' if FLAVOR=='Production (normal Veyra)' else 'dev'
        folder=PROJECT/'app/build/outputs/apk'/flavor/'release'; candidates=list(folder.glob('*.apk'))
        marker={'ARM64 (my Realme phone)':'arm64-v8a','Universal (larger APK)':'universal','ARMv7':'armeabi-v7a','x86_64 (emulator)':'x86_64'}[ARCHITECTURE]
        candidates=[p for p in candidates if marker in p.name]
        if len(candidates)!=1: raise RuntimeError('Expected one '+marker+' APK; found '+str(len(candidates))+'. Inspect the project output names.')
        apk=candidates[0]
        with zipfile.ZipFile(apk) as z:
            if z.testzip() is not None: raise RuntimeError('APK archive integrity failed.')
        _,badging=run([bt/'aapt2','dump','badging',apk],'Read APK package/version',env=env,quiet=True)
        if 'application-debuggable' in badging: raise RuntimeError('Release output unexpectedly has the debuggable flag.')
        info=re.search(r"package: name='([^']+)' versionCode='([^']+)' versionName='([^']+)'",badging)
        if not info: raise RuntimeError('Could not read packaged version metadata.')
        package,code,version=info.groups()
        expected_package='app.veyra.music' if FLAVOR=='Production (normal Veyra)' else 'app.veyra.music.dev'
        if package!=expected_package: raise RuntimeError('Unexpected app package: '+package)
        with zipfile.ZipFile(apk) as archive:
            native_abis={name.split('/')[1] for name in archive.namelist() if name.startswith('lib/') and name.endswith('.so')}
        if marker!='universal' and native_abis!={marker}: raise RuntimeError('APK native libraries do not match selected architecture.')
        if marker=='universal' and not {'arm64-v8a','armeabi-v7a','x86_64'}.issubset(native_abis): raise RuntimeError('Universal APK is missing a configured native architecture.')
        if INFO['version']!='unknown' and version!=INFO['version']: raise RuntimeError('Packaged version differs from source metadata.')
        if INFO['version_code'] and code!=INFO['version_code']: raise RuntimeError('Packaged version code differs from source metadata.')
        certificate=None
        if SIGNING is not None:
            _,verified=run([JAVA/'bin/java','-jar',bt/'lib/apksigner.jar','verify','--verbose','--print-certs',apk],
                           'Verify APK signature',env=env,quiet=True)
            if 'Verifies' not in verified: raise RuntimeError('APK signature did not verify.')
            cert=re.search(r'certificate SHA-256 digest: (\S+)',verified)
            if not cert: raise RuntimeError('Could not read signing certificate fingerprint.')
            certificate=cert[1].lower()
            if certificate!=SIGNING['certificate_sha256']:
                raise RuntimeError('APK was not signed with the key you selected.')
            expected=EXPECTED_CERT_SHA256.replace(':','').strip().lower()
            if expected and certificate!=expected: raise RuntimeError('Signing certificate differs from the expected fingerprint. Do not publish this as an update.')
        RESULT['source_archive_unchanged']=sha(SOURCE_ZIP)==SOURCE_SHA
        changed=[name for name,digest in SOURCE_MANIFEST.items() if not (PROJECT/name).is_file() or sha(PROJECT/name)!=digest]
        RESULT['input_source_files_checked']=len(SOURCE_MANIFEST)
        RESULT['input_source_files_changed']=changed
        if changed: raise RuntimeError('Input source files changed during the build; inspect the report before distributing this APK.')
        if not RESULT['source_archive_unchanged']: raise RuntimeError('Original source ZIP changed unexpectedly.')
        safe_version=re.sub(r'[^A-Za-z0-9._-]','_',version)
        name='Veyra-'+safe_version+'-'+marker+'-release'+('-UNSIGNED' if SIGNING is None else '')+'.apk'
        target=OUTPUT/name; shutil.copy2(apk,target)
        artifact={'name':name,'bytes':target.stat().st_size,'sha256':sha(target),'package':package,'version':version,'version_code':code,
                  'native_abis':sorted(native_abis),'signature_verified':SIGNING is not None,'certificate_sha256':certificate,'archive_integrity':'passed'}
        RESULT['artifacts'].append(artifact)
        (OUTPUT/'SHA256SUMS.txt').write_text(artifact['sha256']+'  '+name+'\n')
        RESULT['status']='ready' if SIGNING is not None else 'unsigned_not_installable'
        if RESULT['tests']['status']=='failed_or_blocked': RESULT['status']+='__tests_failed_or_blocked'
        print('\nAPK READY:',name)
        print('Version:',version,'| code:',code,'| package:',package)
        print('Certificate SHA-256:',certificate or 'UNSIGNED')
        print('Tests:',RESULT['tests']['status'])
        print('No app code was automatically patched. Phone testing remains necessary.')
    except Exception as exc:
        RESULT['status']='failed'; RESULT['error']=clean(exc)
        print('\nBUILD STOPPED:',clean(exc))
        print('Download the results ZIP in the next cell for the log and report. Source errors and repository outages are not silently bypassed.')
    finally:
        (OUTPUT/'build-report.json').write_text(json.dumps(RESULT,indent=2))
        RESULT_ZIP=zip_results()
    return RESULT


## Step 1 · Choose your build
Use the Colab form fields in the next cell. No hidden source-URL prompt is required.

- **SOURCE_CHOICE:** Preview, Stable, custom public GitHub ZIP URL, or upload a source ZIP in Step 2.
- **SOURCE_SHA256:** optional expected source checksum for uploads/custom links. Official presets fetch and enforce their published checksum sidecar.
- **SIGNING_MODE:** keep **Upload existing key or backup** for updates. The default certificate fingerprint matches the published 0.1.5 APK; old TEST backups will fail this check.
- **ARCHITECTURE:** ARM64 for your phone; Universal for all configured native architectures.
- **FLAVOR:** Production replaces the normal app when package, key and Android version rules permit; Dev installs separately.
- Existing-key passwords are loaded from the private ZIP, or entered via hidden prompts. Never enter passwords into form fields.
- New-key and unsigned modes remain available for intentional separate use; clear the existing-release fingerprint only for those modes.

Tests are optional and reported separately from APK assembly. R8 is always enabled. Select a larger heap only when your runtime has sufficient RAM.

[Android SDK terms](https://developer.android.com/studio/terms) · [Colab limits](https://research.google.com/colaboratory/faq.html)


In [ ]:
# @title Step 1 — Build options
ACCEPT_ANDROID_LICENSES = False # @param {type:"boolean"}
SOURCE_CHOICE = "Stable 0.1.8" # @param ["Preview 0.1.9-beta1", "Stable 0.1.8", "Custom GitHub ZIP URL", "Upload source ZIP"]
CUSTOM_SOURCE_URL = "" # @param {type:"string"}
SOURCE_SHA256 = "" # @param {type:"string"}
SIGNING_MODE = "Upload existing key or backup" # @param ["Upload existing key or backup", "Create new key (new installs)", "Unsigned release"]
ARCHITECTURE = "ARM64 (my Realme phone)" # @param ["ARM64 (my Realme phone)", "Universal (larger APK)", "ARMv7", "x86_64 (emulator)"]
FLAVOR = "Production (normal Veyra)" # @param ["Production (normal Veyra)", "Dev (separate install)"]
RUN_TESTS = False # @param {type:"boolean"}
SAVE_RESULTS_TO_DRIVE = False # @param {type:"boolean"}
JAVA_VERSION = 17 # @param [17, 21] {type:"raw"}
WORKERS = 2 # @param [1, 2] {type:"raw"}
BUILD_TIMEOUT_MINUTES = 180 # @param [60, 180, 360] {type:"raw"}
HEAP_MB = 3072 # @param [3072, 4096, 6144] {type:"raw"}
BUILD_TOOLS_OVERRIDE = "" # @param {type:"string"}
EXPECTED_CERT_SHA256 = "22b5a611d9c62ded21114c28074f4f25c378c2ee70866740623e49221198cb9a" # @param {type:"string"}

if not ACCEPT_ANDROID_LICENSES:
    raise ValueError('Read the Android SDK terms, tick ACCEPT_ANDROID_LICENSES in this form, then run this cell again.')
if WORKERS not in {1,2} or BUILD_TIMEOUT_MINUTES not in {60,180,360}:
    raise ValueError("Choose a valid worker count and build timeout.")
if JAVA_VERSION not in {17,21} or HEAP_MB not in {3072,4096,6144}:
    raise ValueError('Choose one of the available Java/heap options.')
if EXPECTED_CERT_SHA256 and not re.fullmatch(r'[a-fA-F0-9]{64}',EXPECTED_CERT_SHA256.replace(':','').strip()):
    raise ValueError('Expected certificate fingerprint must contain 64 hexadecimal characters, or be blank.')
import google.colab
BASE=Path('/content/veyra_builder');TOOLS=BASE/'tools'
BASE.mkdir(parents=True,exist_ok=True);TOOLS.mkdir(parents=True,exist_ok=True)
RUN=BASE/'runs'/(datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')+'-'+uuid.uuid4().hex[:6])
RUN.mkdir(parents=True);OUTPUT=RUN/'output';OUTPUT.mkdir();SECRETS.clear()
RESULT={};RESULT_ZIP=None;SIGNING=None;KEY_ENV={};NEW_KEY_BACKUP=None
presets={'Stable 0.1.8':DEFAULT_URL,'Preview 0.1.9-beta1':PREVIEW_URL}
if SOURCE_CHOICE not in {*presets,'Custom GitHub ZIP URL','Upload source ZIP'}:
    raise ValueError('Choose an available source option.')
SOURCE_URL=presets.get(SOURCE_CHOICE) or (normalize_url(CUSTOM_SOURCE_URL) if SOURCE_CHOICE=='Custom GitHub ZIP URL' else 'local-upload')
if SOURCE_SHA256 and not re.fullmatch(r'[a-fA-F0-9]{64}',SOURCE_SHA256.strip()):
    raise ValueError('Source SHA-256 must be blank or 64 hexadecimal characters.')
if SIGNING_MODE=='Create new key (new installs)' and EXPECTED_CERT_SHA256.strip():
    raise ValueError('A NEW key cannot match the existing-release fingerprint. Only for intentional new installs: clear EXPECTED_CERT_SHA256, then rerun. Existing updates must use the private backup.')
if SIGNING_MODE=='Unsigned release' and EXPECTED_CERT_SHA256.strip():
    raise ValueError('Unsigned output cannot verify a signing certificate. Clear EXPECTED_CERT_SHA256 only if you intentionally need an unsigned APK.')
VARIANT=('Prod' if FLAVOR=='Production (normal Veyra)' else 'Dev')+'Release'
print('Run folder:',RUN)
print('Source:',SOURCE_URL)
print('Variant:',VARIANT,'| Architecture:',ARCHITECTURE)
print('Tests:', 'requested' if RUN_TESTS else 'NOT REQUESTED')

print('Signing certificate check:',EXPECTED_CERT_SHA256 or 'NOT SET — update compatibility is not enforced')


## Step 2 · Get and inspect the source
Official presets download a fresh source ZIP and enforce its published SHA-256 sidecar. A custom GitHub URL or direct source-ZIP upload is also supported; provide SOURCE_SHA256 in Step 1 to verify those inputs against a known digest.

The archive is checked before extraction into a fresh project directory. Source version/code, SDK/NDK/CMake requirements and a source-file fingerprint are recorded. The notebook does not edit application code or change its version. Build only source you trust: Gradle build files execute code in this runtime.


In [ ]:
# @title Step 2 — Download latest source ZIP
SOURCE_ZIP=RUN/'source.zip'
if SOURCE_CHOICE=='Upload source ZIP':
    from google.colab import files
    incoming=RUN/'source-upload';incoming.mkdir(exist_ok=True)
    previous=Path.cwd()
    try:
        os.chdir(incoming);uploaded=files.upload()
    finally:
        os.chdir(previous)
    if len(uploaded)!=1: raise ValueError('Upload exactly one Android SOURCE ZIP, not your private signing ZIP.')
    filename,raw=next(iter(uploaded.items()))
    if not 0<len(raw)<=512*1024**2: raise ValueError('Source upload must be nonempty and at most 512 MiB.')
    SOURCE_ZIP.write_bytes(raw);del raw,uploaded
else:
    download(SOURCE_URL,SOURCE_ZIP,fresh=True)
expected_source=SOURCE_SHA256.strip().lower()
if SOURCE_CHOICE in {'Stable 0.1.8','Preview 0.1.9-beta1'}:
    sidecar=download(SOURCE_URL+'.sha256',RUN/'source.sha256',fresh=True).read_text()
    match=re.match(r'^([a-fA-F0-9]{64})\s',sidecar)
    if not match: raise ValueError('Invalid official source checksum file.')
    published=match[1].lower()
    if expected_source and expected_source!=published: raise ValueError('Provided checksum differs from published checksum.')
    expected_source=published
if expected_source and sha(SOURCE_ZIP)!=expected_source:
    raise ValueError('Source checksum mismatch. Stop and check the selected ZIP/link; do not bypass it.')
print('Source checksum:', 'MATCHED' if expected_source else 'Recorded only — no expected digest provided')
project_dir=RUN/'project'
if project_dir.exists(): shutil.rmtree(project_dir)
if not zipfile.is_zipfile(SOURCE_ZIP):
    raise ValueError('The URL did not return a ZIP. Check that the link is public and points to the source ZIP, not an HTML/login page.')
extract_zip(SOURCE_ZIP,RUN/'project')
PROJECT=project_root(RUN/'project');SOURCE_SHA=sha(SOURCE_ZIP)
INFO=detect(PROJECT);SOURCE_MANIFEST=source_fingerprint(PROJECT)
print('Project:',PROJECT)
print('Version:',INFO['version'],'| code:',INFO['version_code'])
print('Required Android packages:',', '.join(INFO['packages']))
print('Source ZIP SHA-256:',SOURCE_SHA)
print('Input code/resource/build files tracked:',len(SOURCE_MANIFEST))
(OUTPUT/'source-info.json').write_text(json.dumps({'url':SOURCE_URL,'sha256':SOURCE_SHA,'expected_sha256':expected_source or None,'checksum_verified':bool(expected_source),**INFO},indent=2))


## Step 3 · Install or reuse the build tools
Installs a JDK if missing, pinned Android command-line tools, and the project SDK/NDK/CMake packages. The Gradle wrapper downloads the project-pinned Gradle distribution during the build.

Android tools are downloaded from Google's repository and validated after installation. Command-line tools are checked against the checksum in Google's package metadata. An extra nested NDK archive root is repaired only when detected.

In [ ]:
# @title Step 3 — Prepare Java and Android tools
setup_tools()


## Step 4 · Signing
**For an update, upload the same key used for the installed app.** Supported uploads:

- One `.jks` or `.keystore` file: enter its alias and hidden passwords when prompted.
- A portable Veyra signing-backup `.zip`: one keystore plus `signing/credentials.json`; credentials are loaded without printing them.

A Windows-account-encrypted profile alone is not a portable signing backup. Incorrect credentials stop signing; the notebook never silently substitutes another key.

If you explicitly chose **Create new key**, save the **PRIVATE signing backup** using the button before leaving the runtime. A new key will not update installations signed with a different key. Signing uploads and passwords stay out of the public results ZIP. Do not share your private backup or an executed notebook containing sensitive outputs.

In [ ]:
# @title Step 4 — Load your signing key
configure_signing()


## Step 5 · Build optimized Release and verify
R8 and resource shrinking are explicitly enabled without removing project keep rules. The notebook checks the selected R8 task, mapping and effective configuration, then verifies APK integrity, version and signing certificate. Cached successful R8 output is accepted with its reports.

The console reports elapsed time and the last observed task during quiet periods. It cannot promise a finish time. Failures remain failures; optimization is never disabled automatically to make a build pass.

Tests run only if you enabled RUN_TESTS in Step 1. Building an APK does not establish that it is bug-free on your phone.


In [ ]:
# @title Step 5 — Build APK
RESULT=build_apk()
print('\nFinal status:',RESULT['status'])


## Step 6 · Optional copy to Google Drive
If selected in Step 1, authorize your Drive when prompted. Only public build outputs and logs are copied to `MyDrive/Veyra_Builds`. The private signing directory is excluded. Builds run on the local Colab disk for speed.

In [ ]:
# @title Step 6 — Save public results to Drive (optional)
if SAVE_RESULTS_TO_DRIVE:
    if RESULT_ZIP is None:
        RESULT_ZIP=zip_results()
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_target=Path('/content/drive/MyDrive/Veyra_Builds')/RUN.name
        drive_target.mkdir(parents=True,exist_ok=True)
        for p in OUTPUT.rglob('*'):
            if p.is_file():
                (drive_target/p.relative_to(OUTPUT)).parent.mkdir(parents=True,exist_ok=True)
                shutil.copy2(p,drive_target/p.relative_to(OUTPUT))
        shutil.copy2(RESULT_ZIP,drive_target/RESULT_ZIP.name)
        print('Saved public build outputs:',drive_target)
    except Exception as exc:
        print('Drive copy failed; local downloads remain available:',clean(exc))
else:
    print('Drive saving is off. Download your APK and results in Step 7 before the runtime is deleted.')


## Step 7 · Download your APK and build report
Click the buttons below. **Veyra_Build_Results.zip** contains the selected APK (if verified), a log, the source URL/checksum, and a build report. It never includes the private signing directory.

If an earlier cell stopped with an error, you can run this cell manually to package the log and available diagnostics. No successful APK is implied by the presence of a results ZIP.

In [ ]:
# @title Step 7 — Download results
from IPython.display import display, HTML
if 'RUN' not in globals():
    raise RuntimeError('Run Step 1 first.')
if not RESULT:
    RESULT={'status':'incomplete','message':'A previous cell did not complete. See build.log if available.'}
    (OUTPUT/'build-report.json').write_text(json.dumps(RESULT,indent=2))
RESULT_ZIP=zip_results()
status=html.escape(str(RESULT.get('status','unknown')))
display(HTML('<h3>Build status: '+status+'</h3>'))
for artifact in RESULT.get('artifacts',[]):
    path=OUTPUT/artifact['name']
    display(HTML('<p><b>'+html.escape(artifact['name'])+'</b><br>SHA-256: <code>'+html.escape(artifact['sha256'])+'</code></p>'))
    download_button(path,'Download '+artifact['name'])
download_button(RESULT_ZIP,'Download APK + log + report ZIP')
if (RUN/'build.log').is_file():
    download_button(RUN/'build.log','Download build.log')
if RESULT.get('tests',{}).get('status')=='failed_or_blocked':
    print('Tests failed or were blocked. The results report does not mark them as passed.')
if RESULT.get('status','').startswith('unsigned'):
    print('This output is UNSIGNED and cannot be installed until signed.')


## Optional · Remove signing secrets from this runtime
After downloading any new private signing backup and your build results, click **Remove private signing files**. This preserves APKs/logs but removes uploaded/generated signing files and clears signing credentials from the notebook's Python variables. This is best-effort runtime cleanup, not secure erasure of RAM or remote storage.

To build again, rerun from Step 1 and upload the key again. Before sharing this notebook, use **Edit → Clear all outputs** and never attach a private signing backup.

In [ ]:
# @title Optional — Remove private signing files
import ipywidgets as widgets
from IPython.display import display
cleanup_button=widgets.Button(description='Remove private signing files',button_style='warning',layout=widgets.Layout(width='auto'))
cleanup_output=widgets.Output()
def remove_signing(_):
    global SIGNING,KEY_ENV,NEW_KEY_BACKUP
    with cleanup_output:
        shutil.rmtree(RUN/'private',ignore_errors=True)
        KEY_ENV.clear();SIGNING=None;NEW_KEY_BACKUP=None;SECRETS.clear()
        print('Private signing files removed from this run. Public APKs and build reports remain.')
cleanup_button.on_click(remove_signing)
display(cleanup_button,cleanup_output)


## Troubleshooting and reuse

| Symptom | What to do |
| --- | --- |
| First cell pauses | Choose a source preset in Step 1; custom links belong in CUSTOM_SOURCE_URL. |
| SDK licence checkbox error | Tick the box after reviewing the terms, then rerun Step 1 onward. |
| Signing alias/password error | Upload the correct existing portable backup or supply the original key credentials. Do not create a replacement key for an update. |
| Missing `source.properties` | The setup cell attempts the known nested-directory repair. If validation still fails, remove/reinstall the affected local SDK package and rerun setup. |
| JitPack/Google/Maven HTTP failure | Retry after checking service/network availability. The notebook does not substitute unverified dependencies or hide download failures. |
| Kotlin source error | Download the log; fix the source project and provide its new ZIP URL. The notebook does not automatically alter app logic. |
| `test…UnitTest` fails | Inspect the report/log. APK assembly and unit-test success are reported separately. |
| Out of Java heap / daemon disappears | Start a fresh run with a larger heap only if runtime RAM permits; a GPU does not fix Java memory pressure. |
| No phone download starts | Click the download button again or use the Colab Files sidebar to download from the displayed run folder. |
| Colab session was deleted | Rerun the notebook. Tools must be downloaded again; retrieve saved outputs from Drive if enabled. |

**Next version:** Run from Step 1 again and choose a preset or supply its custom public source ZIP URL. A new run folder is created; existing tools/cache in the same runtime are reused. This targets Veyra's current Gradle/Kotlin Android project structure, not arbitrary Android projects or every possible future toolchain change.

### References

- [Android SDK command-line package management](https://developer.android.com/tools/sdkmanager)
- [Android app signing and update identity](https://developer.android.com/studio/publish/app-signing)
- [APK signature verification](https://developer.android.com/tools/apksigner)
- [Colab runtime lifetime and limits](https://research.google.com/colaboratory/faq.html)

**Notebook validation:** code syntax, notebook schema and targeted helper tests are checked during creation. A full Colab APK build is not claimed to have been executed while creating this file. Your Colab runtime performs the actual build.

**Wrong old TEST key:** keep the default release fingerprint and upload `Veyra_PRIVATE_Signing_Backup.zip`. Do not clear the check to force an incompatible update.

**Android refuses an older version:** Android normally blocks older version codes; stable 0.1.6 code 22 can update preview code 21 using the same signing key. Do not uninstall just to bypass this; use the correct newer build or the separate Dev flavor for testing.
